In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/size_only/validation/gp_accuracy'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Evaluate The 1,024-Location Soay Transformer

This notebook evaluates the targeted-prior Transformer DeepRV checkpoint against exact RBF Gaussian-process draws. Every paired comparison uses the same locations, lengthscale, covariance matrix, and standard-normal vector. The Transformer is called with the same 1,024-location padding and active-location mask used during IBM simulation.

The notebook retains three outputs:

1. A table of diagnostics using MCMC lengthscale samples.
2. Scatter plots of performance across posterior lengthscales.
3. Representative first-year exact and Transformer draws.

In [ ]:
from pathlib import Path
import json
import sys

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

current_path = Path.cwd().resolve()
repo_root = PROJECT_ROOT
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from experiments.soay.size_only.training.deeprv_decoder import DeepRVDecoder, JITTER

plt.style.use("seaborn-v0_8-white")
print("Repository Root:", repo_root)
print("JAX Devices:", jax.devices())

## Load The Best Checkpoint

`model.ckpt` stores the state with the best validation loss. The metadata validation loss provides an absolute reference for the independent held-out test.

In [ ]:
checkpoint_folder = GENERATED / 'soay/size_only/training/transformer'
checkpoint_path = checkpoint_folder / "model.ckpt"
metadata_path = checkpoint_folder / "metadata.json"
output_directory = checkpoint_folder / "evaluation_outputs"
output_directory.mkdir(exist_ok=True)

with metadata_path.open() as metadata_file:
    checkpoint_metadata = json.load(metadata_file)

decoder = DeepRVDecoder(checkpoint_path)
assert decoder.max_locations == 1024, f"Expected 1,024 locations, found {decoder.max_locations}"

checkpoint_mse = float(checkpoint_metadata["evaluation_norm_mse"])
checkpoint_rmse = np.sqrt(checkpoint_mse)
print("Checkpoint:", checkpoint_path)
print("Maximum Locations:", decoder.max_locations)
print(f"Stored Validation MSE: {checkpoint_mse:.6f}")
print(f"Stored Validation RMSE: {checkpoint_rmse:.4f}")
print(f"Training Time: {checkpoint_metadata['training_time_seconds'] / 3600:.2f} Hours")

## Evaluation Settings

Increase the repetition counts for a more precise final assessment. The defaults balance posterior coverage and local runtime.

In [ ]:
evaluation_seed = 20260817
posterior_repeats = 25
posterior_draws = 5

rng = np.random.default_rng(evaluation_seed)

## Exact And Transformer Draws

The exact target is

\[
f = \operatorname{Cholesky}\!\left(K_\ell(s,s)+\epsilon I\right)z.
\]

Inactive locations are zero-padded to reproduce deployed decoder behaviour.

In [ ]:
def rbf_covariance(locations, lengthscale):
    locations = np.asarray(locations, dtype=np.float32).reshape(-1)
    distances = locations[:, None] - locations[None, :]
    return np.exp(-0.5 * (distances / lengthscale)**2).astype(np.float32)


def evaluate_draws(locations, lengthscale, latent_z, padded_locations=None):
    locations = np.asarray(locations, dtype=np.float32).reshape(-1)
    latent_z = np.asarray(latent_z, dtype=np.float32)
    num_draws, num_locations = latent_z.shape
    if num_locations != len(locations):
        raise ValueError("The latent vector and locations must have the same length")
    if num_locations > decoder.max_locations:
        raise ValueError(f"At most {decoder.max_locations} locations are allowed")

    if padded_locations is None:
        padded_locations = np.zeros(decoder.max_locations, dtype=np.float32)
        padded_locations[:num_locations] = locations
    else:
        padded_locations = np.asarray(padded_locations, dtype=np.float32).reshape(-1)
        if len(padded_locations) != decoder.max_locations or not np.allclose(padded_locations[:num_locations], locations):
            raise ValueError("Padded locations must contain the active locations first")

    mask = np.arange(decoder.max_locations) < num_locations
    padded_z = np.zeros((num_draws, decoder.max_locations), dtype=np.float32)
    padded_z[:, :num_locations] = latent_z
    padded_covariance = rbf_covariance(padded_locations, lengthscale) + JITTER * np.eye(decoder.max_locations, dtype=np.float32)
    active_covariance = rbf_covariance(locations, lengthscale) + JITTER * np.eye(num_locations, dtype=np.float32)
    exact = np.einsum("ij,bj->bi", np.linalg.cholesky(active_covariance), latent_z)

    predicted = decoder.decode(
        jnp.asarray(padded_z),
        jnp.asarray(lengthscale, dtype=jnp.float32),
        jnp.asarray(padded_locations)[:, None],
        jnp.asarray(padded_covariance),
        jnp.repeat(jnp.asarray(mask)[None], num_draws, axis=0),
    )[:, :num_locations, 0]
    predicted.block_until_ready()
    return exact, np.asarray(jax.device_get(predicted))


def calculate_metrics(exact, predicted, **labels):
    rows = []
    for draw_index, (exact_draw, predicted_draw) in enumerate(zip(exact, predicted)):
        exact_variance = np.var(exact_draw)
        predicted_variance = np.var(predicted_draw)
        correlation = np.corrcoef(exact_draw, predicted_draw)[0, 1] if exact_variance > 0 and predicted_variance > 0 else np.nan
        rows.append({
            **labels,
            "Draw": draw_index,
            "RMSE": np.sqrt(np.mean((exact_draw - predicted_draw)**2)),
            "Correlation": correlation,
            "Variance Ratio": predicted_variance / exact_variance if exact_variance > 0 else np.nan,
            "Bias": np.mean(predicted_draw - exact_draw),
        })
    return rows


def style_axis(axis):
    axis.tick_params(labelsize=9, direction="out")
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)

## Soay MCMC-Sample Evaluation

The table evaluates the full dataset and first year. The scatter plot evaluates the full dataset and all four individual years. Lengthscales are sampled from each vital rate's MCMC output, with several independent standard-normal vectors evaluated per lengthscale.

In [ ]:
gp_data = pd.read_csv(GENERATED / 'soay/size_only/data/gp_ibm_dataset.csv')
with np.load(GENERATED / 'soay/size_only/inference/gpdf_gp_samples.npz') as saved_samples:
    posterior_lengthscales = {
        "Survival": saved_samples["surv_ls"].copy(),
        "Growth": saved_samples["growth_ls"].copy(),
        "Reproduction": saved_samples["repr_ls"].copy(),
        "Recruit Size": saved_samples["rcsz_ls"].copy(),
    }

first_year = gp_data["yr"].eq(1)
vital_rate_locations = {
    "Survival": {
        "Full Dataset": gp_data["z"].dropna().to_numpy(),
        "First Year": gp_data.loc[first_year, "z"].dropna().to_numpy(),
    },
    "Growth": {
        "Full Dataset": gp_data.loc[gp_data["z1"].notna(), "z"].to_numpy(),
        "First Year": gp_data.loc[first_year & gp_data["z1"].notna(), "z"].to_numpy(),
    },
    "Reproduction": {
        "Full Dataset": gp_data.loc[gp_data["Repr"].notna(), "z"].to_numpy(),
        "First Year": gp_data.loc[first_year & gp_data["Repr"].notna(), "z"].to_numpy(),
    },
    "Recruit Size": {
        "Full Dataset": gp_data.loc[gp_data["Rcsz"].notna(), "z"].to_numpy(),
        "First Year": gp_data.loc[first_year & gp_data["Rcsz"].notna(), "z"].to_numpy(),
    },
}

vital_rate_order = ["Survival", "Growth", "Reproduction", "Recruit Size"]
dataset_order = ["Full Dataset", "First Year"]
for vital_rate in vital_rate_order:
    print(f"{vital_rate}: Full Dataset = {len(vital_rate_locations[vital_rate]['Full Dataset'])}, First Year = {len(vital_rate_locations[vital_rate]['First Year'])}")

In [ ]:
number_lengthscale_samples = 5000
draws_per_lengthscale = 1
mse_evaluation_seed = 20260824

mse_rng = np.random.default_rng(mse_evaluation_seed)
mse_rows = []

for vital_rate in vital_rate_order:
    print(f"Evaluating {vital_rate}")

    locations = np.asarray(
        vital_rate_locations[vital_rate]["Full Dataset"],
        dtype=np.float32,
    )

    lengthscale_samples = np.asarray(
        posterior_lengthscales[vital_rate]
    ).reshape(-1)

    lengthscale_samples = lengthscale_samples[
        np.isfinite(lengthscale_samples)
    ]

    sample_indices = mse_rng.choice(
        len(lengthscale_samples),
        size=number_lengthscale_samples,
        replace=number_lengthscale_samples > len(lengthscale_samples),
    )

    for sample_number, sample_index in enumerate(sample_indices):
        lengthscale = float(lengthscale_samples[sample_index])

        latent_z = mse_rng.standard_normal(
            (draws_per_lengthscale, len(locations))
        ).astype(np.float32)

        exact, predicted = evaluate_draws(
            locations,
            lengthscale,
            latent_z,
        )

        mse_per_draw = np.mean((exact - predicted) ** 2, axis=1)

        mse_rows.append({
            "Vital Rate": vital_rate,
            "Lengthscale Sample": sample_number,
            "Lengthscale": lengthscale,
            "Mean MSE": mse_per_draw.mean(),
            "Number Of Locations": len(locations),
            "Draws": draws_per_lengthscale,
        })

lengthscale_mse_results = pd.DataFrame(mse_rows)

In [ ]:
def evaluate_lengthscale_mse_10_draws(
    vital_rates,
    locations_by_vital_rate,
    lengthscales_by_vital_rate,
    seed=20260824,
):
    evaluation_rng = np.random.default_rng(seed)
    result_rows = []
    draws_per_sample = 10

    for rate in vital_rates:
        print(f"Evaluating {rate}")

        rate_locations = np.asarray(
            locations_by_vital_rate[rate]["Full Dataset"],
            dtype=np.float32,
        )

        rate_lengthscales = np.asarray(
            lengthscales_by_vital_rate[rate]
        ).reshape(-1)

        rate_lengthscales = rate_lengthscales[
            np.isfinite(rate_lengthscales)
        ]

        for sample_number, rate_lengthscale in enumerate(rate_lengthscales):
            latent_draws = evaluation_rng.standard_normal(
                (draws_per_sample, len(rate_locations))
            ).astype(np.float32)

            exact_draws, predicted_draws = evaluate_draws(
                rate_locations,
                float(rate_lengthscale),
                latent_draws,
            )

            mse_per_draw = np.mean(
                (exact_draws - predicted_draws) ** 2,
                axis=1,
            )

            rmse_per_draw = np.sqrt(np.mean((exact_draws - predicted_draws) ** 2, axis=1))

            result_rows.append({
                "Vital Rate": rate,
                "Lengthscale Sample": sample_number,
                "Lengthscale": float(rate_lengthscale),
                "Mean MSE": mse_per_draw.mean(),
                "Mean RMSE": rmse_per_draw.mean(),
                "Number Of Locations": len(rate_locations),
                "Draws": draws_per_sample,
            })

    return pd.DataFrame(result_rows)


lengthscale_mse_results_10_draws = evaluate_lengthscale_mse_10_draws(
    vital_rate_order,
    vital_rate_locations,
    posterior_lengthscales,
)

print(f"Total Results: {len(lengthscale_mse_results_10_draws):,}")
display(lengthscale_mse_results_10_draws.head())

In [ ]:
figure, axes = plt.subplots(
    2,
    2,
    figsize=(5.906, 5.2),
    sharey=True,
    constrained_layout=True,
)

for axis, vital_rate in zip(axes.flat, vital_rate_order):
    current = lengthscale_mse_results_10_draws[
        lengthscale_mse_results_10_draws["Vital Rate"] == vital_rate
    ]

    axis.scatter(
        current["Lengthscale"],
        current["Mean RMSE"],
        color="#0072B2",
        s=18,
        alpha=0.1,
    )

    axis.set_title(vital_rate, fontsize=11)
    axis.set_ylim(bottom=0)
    style_axis(axis)

figure.supxlabel("Posterior Lengthscale", fontsize=11)
figure.supylabel("Mean RMSE", fontsize=11)

output_directory = checkpoint_folder / "evaluation_outputs"
output_directory.mkdir(exist_ok=True)

lengthscale_mse_results_10_draws.to_csv(
    output_directory / "mse_across_posterior_lengthscales.csv",
    index=False,
)
figure.savefig(output_directory / "1_dim_Rmse_across_posterior_lengthscales.pdf", bbox_inches="tight",)
figure.savefig(output_directory / "1_dim_Rmse_across_posterior_lengthscales.png", dpi=300, bbox_inches="tight",)

plt.show()

print(f"Saved Outputs: {output_directory}")

In [ ]:
soay_rows = []

for vital_rate in vital_rate_order:
    print(f"Evaluating {vital_rate}")
    lengthscale_samples = posterior_lengthscales[vital_rate]
    sample_indices = rng.integers(0, len(lengthscale_samples), posterior_repeats)

    for repeat, sample_index in enumerate(sample_indices):
        lengthscale = float(lengthscale_samples[sample_index])
        for dataset_name in dataset_order:
            locations = vital_rate_locations[vital_rate][dataset_name]
            latent_z = rng.standard_normal((posterior_draws, len(locations))).astype(np.float32)
            exact, predicted = evaluate_draws(locations, lengthscale, latent_z)
            soay_rows.extend(calculate_metrics(
                exact,
                predicted,
                **{"Vital Rate": vital_rate, "Dataset": dataset_name, "Repeat": repeat, "Locations": len(locations), "Lengthscale": lengthscale},
            ))

soay_results = pd.DataFrame(soay_rows)

summary_table = (
    soay_results
    .groupby(["Dataset", "Vital Rate"], as_index=False)
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        RMSE_5th_Percentile=("RMSE", lambda values: values.quantile(0.05)),
        RMSE_95th_Percentile=("RMSE", lambda values: values.quantile(0.95)),
        Mean_Correlation=("Correlation", "mean"),
        Mean_Variance_Ratio=("Variance Ratio", "mean"),
        Total_Draws=("RMSE", "size"),
    )
)

summary_table["Dataset"] = pd.Categorical(
    summary_table["Dataset"],
    categories=dataset_order,
    ordered=True,
)

summary_table["Vital Rate"] = pd.Categorical(
    summary_table["Vital Rate"],
    categories=vital_rate_order,
    ordered=True,
)

summary_table = summary_table.sort_values(["Dataset", "Vital Rate"]).reset_index(drop=True)
summary_table.to_csv(output_directory / "transformer_diagnostics_table.csv", index=False)

print("Transformer Diagnostics On MCMC Samples")
print(f"Posterior Lengthscale Samples Per Vital Rate: {posterior_repeats}")
print(f"GP Draws Per Lengthscale Sample: {posterior_draws}")
print(f"Total Draws Per Dataset And Vital Rate: {posterior_repeats * posterior_draws}")
print("Saved Table:", output_directory / "transformer_diagnostics_table.csv")

display(
    summary_table.style.format({
        "Mean_RMSE": "{:.4f}",
        "RMSE_5th_Percentile": "{:.4f}",
        "RMSE_95th_Percentile": "{:.4f}",
        "Mean_Correlation": "{:.4f}",
        "Mean_Variance_Ratio": "{:.4f}",
        "Total_Draws": "{:,.0f}",
    })
)

In [ ]:
diagnostic_rng = np.random.default_rng(evaluation_seed + 2)
years = sorted(gp_data["yr"].dropna().unique())

vital_rate_masks = {
    "Survival": gp_data["z"].notna(),
    "Growth": gp_data["z1"].notna(),
    "Reproduction": gp_data["Repr"].notna(),
    "Recruit Size": gp_data["Rcsz"].notna(),
}

diagnostic_locations = {
    "Full Dataset": {
        vital_rate: gp_data.loc[mask, "z"].dropna().to_numpy(dtype=np.float32)
        for vital_rate, mask in vital_rate_masks.items()
    }
}

for year_number, year in enumerate(years, start=1):
    diagnostic_locations[f"Year {year_number}"] = {
        vital_rate: gp_data.loc[mask & gp_data["yr"].eq(year), "z"].dropna().to_numpy(dtype=np.float32)
        for vital_rate, mask in vital_rate_masks.items()
    }

diagnostic_dataset_order = ["Full Dataset"] + [f"Year {year_number}" for year_number in range(1, len(years) + 1)]
diagnostic_rows = []

for vital_rate in vital_rate_order:
    lengthscale_samples = posterior_lengthscales[vital_rate]
    sampled_indices = diagnostic_rng.integers(0, len(lengthscale_samples), size=posterior_repeats)

    for repeat, sample_index in enumerate(sampled_indices):
        lengthscale = float(lengthscale_samples[sample_index])

        for dataset_name in diagnostic_dataset_order:
            locations = diagnostic_locations[dataset_name][vital_rate]

            if len(locations) == 0:
                continue

            latent_z = diagnostic_rng.standard_normal((posterior_draws, len(locations))).astype(np.float32)
            exact, predicted = evaluate_draws(locations, lengthscale, latent_z)

            diagnostic_rows.extend(
                calculate_metrics(
                    exact,
                    predicted,
                    **{
                        "Vital Rate": vital_rate,
                        "Dataset": dataset_name,
                        "Repeat": repeat,
                        "Locations": len(locations),
                        "Lengthscale": lengthscale,
                    },
                )
            )

yearly_soay_results = pd.DataFrame(diagnostic_rows)


In [ ]:
from matplotlib.ticker import MaxNLocator

repeat_results = (
    yearly_soay_results
    .groupby(["Vital Rate", "Dataset", "Repeat", "Lengthscale"], as_index=False)[["RMSE", "Correlation"]]
    .mean()
)

diagnostic_metrics = ["RMSE", "Correlation"]
dataset_colors = {
    "Full Dataset": "#000000",
    "Year 1": "#0072B2",
    "Year 2": "#D55E00",
    "Year 3": "#009E73",
    "Year 4": "#CC79A7",
}

figure, axes = plt.subplots(
    len(vital_rate_order),
    len(diagnostic_metrics),
    figsize=(7.2, 3.0 * len(vital_rate_order)),
    sharex=True,
    sharey="col",
    constrained_layout=True,
)

for row, vital_rate in enumerate(vital_rate_order):
    for column, metric_name in enumerate(diagnostic_metrics):
        axis = axes[row, column]

        for dataset_name in diagnostic_dataset_order:
            subset = repeat_results[
                (repeat_results["Vital Rate"] == vital_rate)
                & (repeat_results["Dataset"] == dataset_name)
            ].sort_values("Lengthscale")

            axis.scatter(
                subset["Lengthscale"],
                subset[metric_name],
                color=dataset_colors[dataset_name],
                s=16,
                alpha=0.6,
                label=dataset_name,
            )

        if row == 0:
            axis.set_title(metric_name, fontsize=11)

        if metric_name == "Correlation":
            axis.axhline(1.0, color="black", linestyle="--", linewidth=1.0, alpha=0.55)

        axis.xaxis.set_major_locator(MaxNLocator(nbins=6))
        axis.tick_params(axis="x", labelbottom=row == len(vital_rate_order) - 1)
        style_axis(axis)

    axes[row, -1].annotate(
        vital_rate,
        xy=(1.05, 0.5),
        xycoords="axes fraction",
        va="center",
        fontsize=10,
    )

handles, labels = axes[0, 0].get_legend_handles_labels()

figure.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=5,
    frameon=False,
)

figure.supxlabel("Posterior Lengthscale", fontsize=14)
figure.suptitle(
    "Transformer Performance Across Posterior Lengthscales",
    fontsize=16,
    y=1.055,
)

figure.savefig(output_directory / "transformer_performance_across_lengthscales.pdf", bbox_inches="tight")
figure.savefig(output_directory / "transformer_performance_across_lengthscales.png", dpi=300, bbox_inches="tight")

plt.show()

## Representative First-Year Draws

Each row uses one vital rate's first-year locations, posterior mean lengthscale, and a reproducible standard-normal vector. The residual panel makes the absolute error scale visible.

In [ ]:
from matplotlib.ticker import MultipleLocator

representative_rng = np.random.default_rng(evaluation_seed + 1)
y_tick_interval = 0.1
draw_values = []
residual_values = []

figure, axes = plt.subplots(
    4,
    2,
    figsize=(8.0, 10.5),
    sharex="col",
    constrained_layout=True,
)

for row, vital_rate in enumerate(vital_rate_order):
    locations = vital_rate_locations[vital_rate]["First Year"]
    lengthscale = float(np.mean(posterior_lengthscales[vital_rate]))
    latent_z = representative_rng.standard_normal((1, len(locations))).astype(np.float32)

    exact, predicted = evaluate_draws(locations, lengthscale, latent_z)
    exact = exact[0]
    predicted = predicted[0]
    residual = predicted - exact

    draw_values.append(np.concatenate([exact, predicted]))
    residual_values.append(residual)

    location_order = np.argsort(locations)
    rmse = np.sqrt(np.mean(residual**2))
    correlation = np.corrcoef(exact, predicted)[0, 1]

    axes[row, 0].scatter(
        locations[location_order],
        exact[location_order],
        color="black",
        s=7,
        alpha=0.7,
        label="Exact",
    )
    axes[row, 0].scatter(
        locations[location_order],
        predicted[location_order],
        color="#0072B2",
        s=6,
        alpha=0.7,
        label="Transformer",
    )
    axes[row, 1].scatter(
        locations[location_order],
        residual[location_order],
        color="#D55E00",
        s=7,
        alpha=0.75,
    )
    axes[row, 1].axhline(
        0.0,
        color="black",
        linestyle="--",
        linewidth=1.0,
        alpha=0.55,
    )
    axes[row, 1].text(
        0.03,
        0.95,
        f"RMSE = {rmse:.3f}\nCorrelation = {correlation:.3f}",
        transform=axes[row, 1].transAxes,
        va="top",
        fontsize=9,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.75,
            "pad": 2,
        },
    )

    if row == 0:
        axes[row, 0].set_title("Exact And Transformer Draw", fontsize=11)
        axes[row, 1].set_title("Transformer Residual", fontsize=11)

    axes[row, 1].annotate(
        f"{vital_rate}\n$n = {len(locations)}$, $\\ell = {lengthscale:.2f}$",
        xy=(1.04, 0.5),
        xycoords="axes fraction",
        va="center",
        fontsize=10,
    )

    style_axis(axes[row, 0])
    style_axis(axes[row, 1])

required_draw_span = max(np.ptp(values) for values in draw_values)
required_residual_span = max(2 * np.max(np.abs(values)) for values in residual_values)
common_y_span = 1.1 * max(required_draw_span, required_residual_span)
common_y_span = np.ceil(common_y_span / y_tick_interval) * y_tick_interval

for row in range(len(vital_rate_order)):
    draw_centre = (draw_values[row].min() + draw_values[row].max()) / 2

    axes[row, 0].set_ylim(
        draw_centre - common_y_span / 2,
        draw_centre + common_y_span / 2,
    )
    axes[row, 1].set_ylim(
        -common_y_span / 2,
        common_y_span / 2,
    )

    for axis in axes[row]:
        axis.yaxis.set_major_locator(MultipleLocator(0.2))
        axis.yaxis.set_minor_locator(MultipleLocator(0.1))
        axis.tick_params(axis="y", which="minor", length=2)

handles, labels = axes[0, 0].get_legend_handles_labels()

size_label = figure.supxlabel("Size", fontsize=14)
figure.supylabel("GP Units", fontsize=14)

figure.suptitle(
    "Representative First-Year GP Draws",
    fontsize=16,
    y=1.08,
)

figure.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False,
)

figure.canvas.draw()

plot_left = axes[-1, 0].get_position().x0
plot_right = axes[-1, 1].get_position().x1
plot_centre = (plot_left + plot_right) / 2
size_label.set_x(plot_centre)

figure.savefig(output_directory / "representative_first_year_draws.pdf", bbox_inches="tight")
figure.savefig(output_directory / "representative_first_year_draws.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ============================================================
# Distributional evaluation settings
# ============================================================

distribution_seed = 20260820

n_lengthscales = 10
n_draws = 100
decoder_batch_size = 32

distribution_rng = np.random.default_rng(distribution_seed)

In [ ]:
def evaluate_distribution(
    locations,
    lengthscale,
    n_draws=1000,
    batch_size=32,
    rng=None,
):
    """
    Generate paired exact-GP and DeepRV draws at fixed locations
    and a fixed lengthscale.

    Returns
    -------
    exact : (n_draws, n_locations)
    predicted : (n_draws, n_locations)
    true_covariance : (n_locations, n_locations)
    """

    if rng is None:
        rng = np.random.default_rng()

    locations = np.asarray(locations, dtype=np.float32).reshape(-1)
    n_locations = len(locations)

    if n_locations > decoder.max_locations:
        raise ValueError(
            f"At most {decoder.max_locations} locations are allowed"
        )

    # --------------------------------------------------------
    # True GP covariance at active locations
    # --------------------------------------------------------

    true_covariance = (
        rbf_covariance(locations, lengthscale)
        + JITTER * np.eye(n_locations, dtype=np.float32)
    )

    L = np.linalg.cholesky(true_covariance)

    # --------------------------------------------------------
    # Draw all z once
    # --------------------------------------------------------

    latent_z = rng.standard_normal(
        (n_draws, n_locations)
    ).astype(np.float32)

    # Exact GP draws: shape (n_draws, n_locations)
    exact = latent_z @ L.T

    # --------------------------------------------------------
    # Prepare fixed padded quantities for DeepRV
    # --------------------------------------------------------

    padded_locations = np.zeros(
        decoder.max_locations,
        dtype=np.float32
    )
    padded_locations[:n_locations] = locations

    mask = np.arange(decoder.max_locations) < n_locations

    padded_covariance = (
        rbf_covariance(padded_locations, lengthscale)
        + JITTER * np.eye(
            decoder.max_locations,
            dtype=np.float32
        )
    )

    # --------------------------------------------------------
    # DeepRV draws in batches
    # --------------------------------------------------------

    predicted_batches = []

    for start in range(0, n_draws, batch_size):

        end = min(start + batch_size, n_draws)

        z_batch = latent_z[start:end]
        current_batch_size = len(z_batch)

        padded_z = np.zeros(
            (current_batch_size, decoder.max_locations),
            dtype=np.float32
        )
        padded_z[:, :n_locations] = z_batch

        predicted_batch = decoder.decode(
            jnp.asarray(padded_z),
            jnp.asarray(lengthscale, dtype=jnp.float32),
            jnp.asarray(padded_locations)[:, None],
            jnp.asarray(padded_covariance),
            jnp.repeat(
                jnp.asarray(mask)[None],
                current_batch_size,
                axis=0,
            ),
        )[:, :n_locations, 0]

        predicted_batch.block_until_ready()

        predicted_batches.append(
            np.asarray(jax.device_get(predicted_batch))
        )

    predicted = np.concatenate(predicted_batches, axis=0)

    return exact, predicted, true_covariance

In [ ]:
def distribution_metrics(exact, predicted, true_covariance):
    """
    Compare the distribution of DeepRV draws with the GP target.
    """

    # --------------------------------------------------------
    # 1. Paired MSE
    # --------------------------------------------------------

    mse_per_draw = np.mean(
        (predicted - exact) ** 2,
        axis=1
    )

    mean_mse = np.mean(mse_per_draw)

    # --------------------------------------------------------
    # 2. Covariance matrices
    #
    # rows = draws
    # columns = locations
    # --------------------------------------------------------

    exact_covariance = np.cov(
        exact,
        rowvar=False,
        ddof=1
    )

    deeprv_covariance = np.cov(
        predicted,
        rowvar=False,
        ddof=1
    )

    # Relative Frobenius covariance error
    deeprv_covariance_error = (
        np.linalg.norm(
            deeprv_covariance - true_covariance,
            ord="fro"
        )
        /
        np.linalg.norm(
            true_covariance,
            ord="fro"
        )
    )

    # Useful baseline:
    # how far does a finite sample of exact GP draws deviate from K?
    gp_sampling_covariance_error = (
        np.linalg.norm(
            exact_covariance - true_covariance,
            ord="fro"
        )
        /
        np.linalg.norm(
            true_covariance,
            ord="fro"
        )
    )

    # --------------------------------------------------------
    # 3. Variance at each location, across draws
    # --------------------------------------------------------

    deeprv_variance = np.var(
        predicted,
        axis=0,
        ddof=1
    )

    exact_variance = np.var(
        exact,
        axis=0,
        ddof=1
    )

    theoretical_variance = np.diag(true_covariance)

    # Main variance ratio
    variance_ratio = (
        deeprv_variance / theoretical_variance
    )

    # Exact-GP Monte Carlo baseline
    exact_variance_ratio = (
        exact_variance / theoretical_variance
    )

    return {
        "Mean MSE": mean_mse,
        "DeepRV Covariance Error": deeprv_covariance_error,
        "GP Sampling Covariance Error": gp_sampling_covariance_error,
        "Variance Ratio": variance_ratio,
        "Exact Variance Ratio": exact_variance_ratio,
        "DeepRV Covariance": deeprv_covariance,
        "Exact Sample Covariance": exact_covariance,
        "True Covariance": true_covariance,
        "MSE Per Draw": mse_per_draw,
    }

In [ ]:
distribution_results = {}
distribution_summary_rows = []

quantiles = np.linspace(0.05, 0.95, n_lengthscales)

for vital_rate in vital_rate_order:

    print(f"Evaluating {vital_rate}")

    locations = vital_rate_locations[vital_rate]["First Year"]

    lengthscale_samples = np.asarray(
        posterior_lengthscales[vital_rate]
    ).reshape(-1)

    selected_lengthscales = np.quantile(
        lengthscale_samples,
        quantiles
    )

    distribution_results[vital_rate] = {}

    for i, lengthscale in enumerate(selected_lengthscales):

        print(
            f"  {i + 1}/{n_lengthscales}: "
            f"lengthscale = {lengthscale:.4f}"
        )

        exact, predicted, true_covariance = evaluate_distribution(
            locations=locations,
            lengthscale=float(lengthscale),
            n_draws=n_draws,
            batch_size=decoder_batch_size,
            rng=distribution_rng,
        )

        metrics = distribution_metrics(
            exact,
            predicted,
            true_covariance
        )

        distribution_results[vital_rate][i] = {
            "Lengthscale": float(lengthscale),
            "Locations": locations,
            **metrics,
        }

        distribution_summary_rows.append({
            "Vital Rate": vital_rate,
            "Lengthscale": float(lengthscale),
            "Mean MSE": metrics["Mean MSE"],
            "DeepRV Covariance Error":
                metrics["DeepRV Covariance Error"],
            "GP Sampling Covariance Error":
                metrics["GP Sampling Covariance Error"],
            "Mean Variance Ratio":
                np.mean(metrics["Variance Ratio"]),
            "Median Variance Ratio":
                np.median(metrics["Variance Ratio"]),
        })


distribution_summary = pd.DataFrame(
    distribution_summary_rows
)

display(distribution_summary)

In [ ]:
for vital_rate in vital_rate_order:

    fig, ax = plt.subplots(figsize=(9, 5))

    for result in distribution_results[vital_rate].values():

        locations = result["Locations"]
        variance_ratio = result["Variance Ratio"]
        lengthscale = result["Lengthscale"]

        order = np.argsort(locations)

        ax.plot(
            locations[order],
            variance_ratio[order],
            alpha=0.7,
            label=f"ℓ = {lengthscale:.2f}"
        )

    ax.axhline(1.0, linestyle="--", linewidth=1)

    ax.set_xlabel("Size")
    ax.set_ylabel("DeepRV variance / GP variance")
    ax.set_title(f"{vital_rate}: variance ratio")

    ax.legend(
        title="Lengthscale",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    style_axis(ax)

    plt.tight_layout()
    plt.show()

In [ ]:
vital_rate = "Growth"
result_index = 5

result = distribution_results[vital_rate][result_index]

deeprv_cov = result["DeepRV Covariance"]
true_cov = result["True Covariance"]
exact_cov = result["Exact Sample Covariance"]

In [ ]:
difference = deeprv_cov - true_cov

fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(
    difference,
    aspect="auto",
    origin="lower"
)

ax.set_title(
    f"{vital_rate}: DeepRV - GP covariance\n"
    f"ℓ = {result['Lengthscale']:.2f}"
)

ax.set_xlabel("Location index")
ax.set_ylabel("Location index")

plt.colorbar(im, ax=ax, label="Covariance difference")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(
    true_cov,
    aspect="auto",
    origin="lower"
)

ax.set_title(
    f"{vital_rate}: True GP covariance\n"
    f"ℓ = {result['Lengthscale']:.2f}"
)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(
    deeprv_cov,
    aspect="auto",
    origin="lower"
)

ax.set_title(
    f"{vital_rate}: DeepRV covariance\n"
    f"ℓ = {result['Lengthscale']:.2f}"
)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
for vital_rate in vital_rate_order:

    lengthscales = []
    covariance_errors = []
    gp_sampling_errors = []

    for result in distribution_results[vital_rate].values():
        lengthscales.append(result["Lengthscale"])
        covariance_errors.append(result["DeepRV Covariance Error"])
        gp_sampling_errors.append(result["GP Sampling Covariance Error"])

    order = np.argsort(lengthscales)

    lengthscales = np.asarray(lengthscales)[order]
    covariance_errors = np.asarray(covariance_errors)[order]
    gp_sampling_errors = np.asarray(gp_sampling_errors)[order]

    fig, ax = plt.subplots(figsize=(7, 4))

    ax.plot(
        lengthscales,
        covariance_errors,
        marker="o",
        label="DeepRV"
    )

    ax.plot(
        lengthscales,
        gp_sampling_errors,
        marker="o",
        linestyle="--",
        label="Exact GP sampling baseline"
    )

    ax.set_xlabel("Lengthscale")
    ax.set_ylabel("Relative covariance error")
    ax.set_title(f"{vital_rate}: covariance error")

    ax.legend()
    style_axis(ax)

    plt.tight_layout()
    plt.show()